In [1]:
#  Ask the operating system what video card is plugged in
!nvidia-smi

# Check if i am using a100 gpu
import torch

if torch.cuda.is_available():
    print(f"Success! Connected to: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU only")

Fri Dec 12 11:11:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 573.24                 Driver Version: 573.24         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              5W /   60W |       0MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from sklearn.model_selection import train_test_split
from scipy.spatial import KDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from pathlib import Path
import pandas as pd
import numpy as np
import os
import glob

In [6]:
# Configuration
bus_file    =      '../../data/clean_data/bus/clean_bus_df.csv'
subway_file =      '../../data/clean_data/sub/cleaned_subway_entrances.csv'
output_file =      '../../data/joined/augmented_properties_with_distances.csv'
chunk_size  =      100000  # Adjust for RAM (smaller = slower but safer)
total_rows  =      0

# Creates joined folder if it doesn't exist
path_obj = Path(output_file)
path_obj.parent.mkdir(parents=True, exist_ok=True)

# Load data
buses = pd.read_csv(bus_file)
subways = pd.read_csv(subway_file)

# Prep coords
bus_coords = np.column_stack((buses['Latitude'], buses['Longitude']))
subway_coords = np.column_stack((subways['Entrance Latitude'], subways['Entrance Longitude']))

# Build KD Trees (very fast look up)
bus_tree = KDTree(bus_coords)
subway_tree = KDTree(subway_coords)

# Clear output if exists
if os.path.exists(output_file):
    os.remove(output_file)

In [7]:
# Haversine formula for distance in meters

def haversine(lon1: np.ndarray, lat1: np.ndarray, lon2: np.ndarray, lat2: np.ndarray) -> float:
    R = 6371000  # Radius of Earth in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

In [8]:
print(f"Processing cleaned_property_part_*.csv files in chunks of {chunk_size}...")

dtypes = {
    'BBL': 'object',
    'BORO': 'Int64',
    'TAXCLASS': 'Int64',
    'FULLVAL': 'Int64',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'NTA': 'category',
    'Borough': 'category'
}


part_files = sorted(glob.glob('../../data/clean_data/val/cleaned_property_part_*.csv'))
print(f"Found {len(part_files)} files to process.")

for part_file in part_files:
    part_num = part_file.split('_')[-1].replace('.csv', '')  # Extract number for logging
    print(f"Starting part {part_num}: {part_file}")
    
    # Process each part file in chunks
    for chunk_idx, chunk in enumerate(pd.read_csv(part_file, chunksize=chunk_size, low_memory=False, dtype=dtypes)):
        print(f"  Chunk {chunk_idx + 1} in part {part_num}: {len(chunk)} rows")
        
        prop_coords = np.column_stack((chunk['Latitude'], chunk['Longitude']))
        
        # Nearest bus
        _, bus_idx = bus_tree.query(prop_coords, k=1)
        chunk['dist_to_bus_m'] = [haversine(chunk.iloc[i]['Latitude'], chunk.iloc[i]['Longitude'],
                                            buses.iloc[idx]['Latitude'], buses.iloc[idx]['Longitude'])
                                  for i, idx in enumerate(bus_idx.flatten())]
        chunk['nearest_bus_nta'] = [buses.iloc[idx]['NTAName'] for idx in bus_idx.flatten()]  # Optional extra
        
        # Nearest subway
        _, sub_idx = subway_tree.query(prop_coords, k=1)
        chunk['dist_to_subway_m'] = [haversine(chunk.iloc[i]['Latitude'], chunk.iloc[i]['Longitude'],
                                               subways.iloc[idx]['Entrance Latitude'], subways.iloc[idx]['Entrance Longitude'])
                                     for i, idx in enumerate(sub_idx.flatten())]
        chunk['nearest_subway_routes'] = [subways.iloc[idx]['Routes'] for idx in sub_idx.flatten()]  # Optional extra
        
        # Select final columns (keep originals + new ones)
        final_cols = ['BBL', 'BORO', 'TAXCLASS', 'FULLVAL', 'Latitude', 'Longitude', 'NTA', 'Borough',
                      'dist_to_bus_m', 'dist_to_subway_m', 'nearest_bus_nta', 'nearest_subway_routes']
        chunk_final = chunk[final_cols]
        
        # Append to output
        mode = 'a' if total_rows > 0 else 'w'
        header = total_rows == 0
        chunk_final.to_csv(output_file, mode=mode, header=header, index=False)
        total_rows += len(chunk_final)

print(f"Done! Output: {output_file} with {total_rows} rows")

Processing cleaned_property_part_*.csv files in chunks of 100000...
Found 8 files to process.
Starting part 1: ../../data/clean_data/val/cleaned_property_part_1.csv
  Chunk 1 in part 1: 100000 rows
  Chunk 2 in part 1: 100000 rows
  Chunk 3 in part 1: 100000 rows
  Chunk 4 in part 1: 100000 rows
  Chunk 5 in part 1: 100000 rows
  Chunk 6 in part 1: 100000 rows
  Chunk 7 in part 1: 100000 rows
  Chunk 8 in part 1: 100000 rows
  Chunk 9 in part 1: 100000 rows
  Chunk 10 in part 1: 100000 rows
Starting part 2: ../../data/clean_data/val/cleaned_property_part_2.csv
  Chunk 1 in part 2: 100000 rows
  Chunk 2 in part 2: 100000 rows
  Chunk 3 in part 2: 100000 rows
  Chunk 4 in part 2: 100000 rows
  Chunk 5 in part 2: 100000 rows
  Chunk 6 in part 2: 100000 rows
  Chunk 7 in part 2: 100000 rows
  Chunk 8 in part 2: 100000 rows
  Chunk 9 in part 2: 100000 rows
  Chunk 10 in part 2: 100000 rows
Starting part 3: ../../data/clean_data/val/cleaned_property_part_3.csv
  Chunk 1 in part 3: 100000 row

In [9]:
# Evaluation  --- ONLY ON THE FIRST 100K

df = pd.read_csv('../../data/joined/augmented_properties_with_distances.csv', low_memory=False)
df['log_fullval'] = np.log(df['FULLVAL'] + 1)

# New features
df['log_dist_bus'] = np.log(df['dist_to_bus_m'] + 1)  # + 1 avoids log(0)
df['log_dist_subway'] = np.log(df['dist_to_subway_m'] + 1)
df['num_subway_routes'] = df['nearest_subway_routes'].str.split(',').str.len().fillna(0)  # Count routes


In [ ]:
def process_data(df):
    # Select expanded X - drop any NaNs after deriving
    X = df[['log_dist_bus', 'log_dist_subway', 'TAXCLASS', 'num_subway_routes', 'NTA']].copy()
    y = df['log_fullval']

    # Handle categories: One hot NTA (top 20 for simplicity --- full would use all)
    top_ntas = X['NTA'].value_counts().head(20).index
    X['NTA'] = X['NTA'].apply(lambda x: x if x in top_ntas else 'Other')

    # Drop NaNs
    mask = ~(X.isna().any(axis=1)) & y.notna()
    X_clean = X[mask]
    y_clean = y[mask]
    print(f"Shape after dropping NaNs: {X_clean.shape} (dropped {len(X) - len(X_clean)} rows)")

    # Pipeline for preprocessing 
    preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(drop='first', sparse_output=False), ['NTA'])], remainder='passthrough')

    # Split and fit pipeline with Random Forest
    X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)
    model = Pipeline([('preproc', preprocessor),('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    model.fit(X_train, y_train)

    # Predict and score
    y_pred = model.predict(X_test)
    print(f"R-squared score on test set: {r2_score(y_test, y_pred):.4f}")

    # Feature importances - see if we can extract any insights
    importances = model.named_steps['rf'].feature_importances_
    feature_names = (['NTA_' + name for name in top_ntas[1:]] +  # Skip first for drop
                    ['log_dist_bus', 'log_dist_subway', 'TAXCLASS', 'num_subway_routes'])
    top_feature_imps = {feature_names[i]: importances[i] for i in range(len(feature_names))}
    print("Top feature importances:", top_feature_imps)

    return r2_score(y_test, y_pred), top_feature_imps

### Full Basis

In [ ]:
# Run model on full dataset
process_data(df)

In [1]:
def save_zone_results(res, territory_type):
    df = pd.DataFrame(columns=['DISTRICT_NAME', 'TERRITORY_TYPE', 'r2_score', 'log_dist_subway', 'log_dist_bus', 'TAXCLASS', 'num_subway_routes'])
    for dist_name, r2, feat_imps in res:
        df[len(df)] = pd.Series({
            'DISTRICT_NAME': dist_name,
            'TERRITORY_TYPE': territory_type,
            'r2_score': r2,
            'log_dist_subway': feat_imps['log_dist_subway'],
            'log_dist_bus': feat_imps['log_dist_bus'],
            'TAXCLASS': feat_imps['TAXCLASS'],
            'num_subway_routes': feat_imps['num_subway_routes']
        })
    df.to_csv(f"../../data/results/{territory_type}_results.csv")

### Per-Zone Basis

In [20]:
df['NTA'].value_counts().head(20).index

Index(['South Ozone Park', 'Canarsie', 'Borough Park', 'Parkchester',
       'Great Kills', 'Park Slope-Gowanus', 'St. Albans',
       'Georgetown-Marine Park-Bergen Beach-Mill Basin', 'East New York',
       'Lincoln Square', 'Queens Village', 'Upper West Side', 'Flatlands',
       'Sheepshead Bay-Gerritsen Beach-Manhattan Beach', 'Bensonhurst West',
       'Bayside-Bayside Hills', 'Bay Ridge',
       'New Springville-Bloomfield-Travis', 'Flushing', 'Middle Village'],
      dtype='object', name='NTA')

In [2]:
# Run model on dataset separated by zone
res_nta = []

for nta in df['NTA'].value_counts().head(20).index:
    df_nta = df[df['NTA'] == nta]
    r2_ret, tfi_ret = process_data(df_nta)
    res_nta.append((nta, r2_ret, sorted(tfi_ret, key=lambda x: x[0])))

save_zone_results(res=res_nta, territory_type="NTA")
res_nta

NameError: name 'df' is not defined

In [25]:
res_nta[:5]

[('South Ozone Park',
  0.7125465911628308,
  [('TAXCLASS', np.float64(0.05933596611097391)),
   ('log_dist_bus', np.float64(0.4625161651704092)),
   ('log_dist_subway', np.float64(0.4730503974659772)),
   ('num_subway_routes', np.float64(0.005097471252639543))]),
 ('Canarsie',
  0.9220821646334503,
  [('TAXCLASS', np.float64(0.3249138537028884)),
   ('log_dist_bus', np.float64(0.22769208848913347)),
   ('log_dist_subway', np.float64(0.4473940578079781)),
   ('num_subway_routes', np.float64(0.0))]),
 ('Borough Park',
  0.8756830506088205,
  [('TAXCLASS', np.float64(0.3287846587586568)),
   ('log_dist_bus', np.float64(0.3399964022979108)),
   ('log_dist_subway', np.float64(0.3254516937849205)),
   ('num_subway_routes', np.float64(0.005767245158512035))]),
 ('Parkchester',
  0.8207668617441591,
  [('TAXCLASS', np.float64(0.7229203458526324)),
   ('log_dist_bus', np.float64(0.14729373358678374)),
   ('log_dist_subway', np.float64(0.12978592056058394)),
   ('num_subway_routes', np.float64(

### Per-Borough Basis

In [22]:
df['Borough'].value_counts().head(20).index

Index(['QUEENS', 'BROOKLYN', 'MANHATTAN', 'STATEN IS', 'BRONX'], dtype='object', name='Borough')

In [ ]:
# Run model on dataset separated by zone
res_bor = []

for bor in df['Borough'].value_counts().index:
    df_bor = df[df['Borough'] == bor]
    r2_ret, tfi_ret = process_data(df_bor)
    res_bor.append((bor, r2_ret, sorted(tfi_ret, key=lambda x: x[0])))

save_zone_results(res=res_bor, territory_type="BOR")
res_bor

Shape after dropping NaNs: (2969523, 5) (dropped 176410 rows)
R-squared score on test set: 0.9213
Top feature importances: [('TAXCLASS', np.float64(0.3187461038905535)), ('log_dist_subway', np.float64(0.28603262072794755)), ('num_subway_routes', np.float64(0.2564936341956194)), ('NTA_Murray Hill', np.float64(0.014013958611984146)), ('NTA_Flushing', np.float64(0.012565066457853605))]
Shape after dropping NaNs: (2465668, 5) (dropped 423208 rows)
R-squared score on test set: 0.9209
Top feature importances: [('num_subway_routes', np.float64(0.3286781642267901)), ('log_dist_subway', np.float64(0.2762375544554546)), ('TAXCLASS', np.float64(0.2677676200166661)), ('NTA_Bushwick South', np.float64(0.016481374425890153)), ('NTA_Dyker Heights', np.float64(0.011973593522688914))]
Shape after dropping NaNs: (1224509, 5) (dropped 116546 rows)
R-squared score on test set: 0.8897
Top feature importances: [('TAXCLASS', np.float64(0.34817396127382344)), ('log_dist_subway', np.float64(0.2895480718582958)

[('QUEENS',
  0.9213195523333926,
  [('NTA_Astoria', np.float64(0.0036373278884611923)),
   ('NTA_Baisley Park', np.float64(0.006097129984720446)),
   ('NTA_Bayside-Bayside Hills', np.float64(0.004328442570834251)),
   ('NTA_Elmhurst', np.float64(0.007165945721921889)),
   ('NTA_Flushing', np.float64(0.012565066457853605)),
   ('NTA_Forest Hills', np.float64(0.00755575387866454)),
   ('NTA_Glendale', np.float64(0.0007281132697957843)),
   ('NTA_Hunters Point-Sunnyside-West Maspeth',
    np.float64(0.0060664911521216384)),
   ('NTA_Jackson Heights', np.float64(0.003636954007376295)),
   ('NTA_Middle Village', np.float64(0.005583445461249351)),
   ('NTA_Murray Hill', np.float64(0.014013958611984146)),
   ('NTA_Queens Village', np.float64(0.0011275895167720976)),
   ('NTA_Richmond Hill', np.float64(0.002428442464244507)),
   ('NTA_Ridgewood', np.float64(0.0037178285197692028)),
   ('NTA_South Jamaica', np.float64(0.0041664253931040325)),
   ('NTA_St. Albans', np.float64(0.0055520365889078

In [34]:
res_bor

[('QUEENS',
  0.9213195523333926,
  [('NTA_Astoria', np.float64(0.0036373278884611923)),
   ('NTA_Baisley Park', np.float64(0.006097129984720446)),
   ('NTA_Bayside-Bayside Hills', np.float64(0.004328442570834251)),
   ('NTA_Elmhurst', np.float64(0.007165945721921889)),
   ('NTA_Flushing', np.float64(0.012565066457853605)),
   ('NTA_Forest Hills', np.float64(0.00755575387866454)),
   ('NTA_Glendale', np.float64(0.0007281132697957843)),
   ('NTA_Hunters Point-Sunnyside-West Maspeth',
    np.float64(0.0060664911521216384)),
   ('NTA_Jackson Heights', np.float64(0.003636954007376295)),
   ('NTA_Middle Village', np.float64(0.005583445461249351)),
   ('NTA_Murray Hill', np.float64(0.014013958611984146)),
   ('NTA_Queens Village', np.float64(0.0011275895167720976)),
   ('NTA_Richmond Hill', np.float64(0.002428442464244507)),
   ('NTA_Ridgewood', np.float64(0.0037178285197692028)),
   ('NTA_South Jamaica', np.float64(0.0041664253931040325)),
   ('NTA_St. Albans', np.float64(0.0055520365889078